# Loading and Combining the King County Data

The dataset ships with this repository as **two tables**:

| File | One row per | Columns |
| :--- | :---------- | :------ |
| `data/king_county_house_sales.csv` | sale | `date`, `price`, `house_id`, `id` |
| `data/king_county_house_details.csv` | house | `id`, `bedrooms`, `bathrooms`, `sqft_living`, and the other property attributes |

Your first task is to combine them into one DataFrame. Replace every `...` in the code cells below, and run the cells in order.

In [4]:
import pandas as pd

sales = pd.read_csv("data/king_county_house_sales.csv")


details = pd.read_csv("data/king_county_house_details.csv")

print(f"sales:   {sales.shape[0]:>6} rows, {sales.shape[1]} columns")
print(f"details: {details.shape[0]:>6} rows, {details.shape[1]} columns")

sales:    21597 rows, 4 columns
details:  21420 rows, 19 columns


## Look Before You Merge

Inspect both tables first. Before writing any merge, answer these three questions:

1. Which column in `sales` refers to a house, and which column in `details` identifies one?
2. Is the relationship one to one, or does one house appear in several sales?
3. How many rows do you expect the combined table to have?

In [5]:
sales.head()

,date,price,house_id,id
0,2014-10-13,221900.0,7129300520,1
1,2014-12-09,538000.0,6414100192,2
2,2015-02-25,180000.0,5631500400,3
3,2014-12-09,604000.0,2487200875,4
4,2015-02-18,510000.0,1954400510,5


In [6]:
details.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,3.0,1.00,1180.0,5650.0,1.0,NaN,0.0,3,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,6414100192,3.0,2.25,2570.0,7242.0,2.0,0.0,0.0,3,7,2170.0,400.0,1951,1991.0,98125,47.7210,-122.319,1690.0,7639.0
2,5631500400,2.0,1.00,770.0,10000.0,1.0,0.0,0.0,3,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2487200875,4.0,3.00,1960.0,5000.0,1.0,0.0,0.0,5,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,1954400510,3.0,2.00,1680.0,8080.0,1.0,0.0,0.0,3,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


In [7]:
# TODO: count the rows and the unique values of the key column in each table.
#       .nunique() counts the unique values of a column.
print(f"sales rows:          {len(sales)}")
print(f"unique house_id:     {sales['house_id'].nunique()}")
print(f"details rows:        {len(details)}")
print(f"unique details id:   {details['id'].nunique()}")

# Wie oft wurde ein Haus höchstens verkauft?
print("max sales per house:", sales["house_id"].value_counts().max())

sales rows:          21597
unique house_id:     21420
details rows:        21420
unique details id:   21420
max sales per house: 3


**Your answers:**

1. house_id in sales verweist auf ein Haus, id in details kennzeichnet es.
2. Eins-zu-viele: Ein Haus kann mehrere Verkäufe haben.
3. Erwartet werden 21.597 Zeilen, eine pro Verkauf.

## Check the Key Before You Merge

A merge connects each row of `sales` with **every** row of `details` that has the same key. You get exactly one row per sale only if each key appears **once** in `details`. If a key appeared twice, every sale of that house would be copied, and the combined table would have more rows than there are sales, without any error or warning.

Check that every `id` appears only once in `details` before you merge.

In [8]:
# TODO: check that every id appears only once in details.
#       The attribute .is_unique of a column is True if no value appears twice.
print("every id is unique:", details["id"].is_unique)

every id is unique: True


## Choose the Type of Merge

[`pd.merge()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) takes a `how` argument that decides which rows are kept when a key appears in only one of the two tables:

| `how` | Keeps |
| :--- | :--- |
| `"inner"` | only the keys that appear in both tables |
| `"left"` | every row of the left table (`sales`), with empty values where `details` has no match |
| `"right"` | every row of the right table (`details`), with empty values where `sales` has no match |
| `"outer"` | every row of both tables |

To choose, find out whether the two tables match completely: are there sales whose house is missing from `details`, and houses in `details` that were never sold?

In [9]:
# TODO: count the sales whose house_id does not appear in details,
#       and the houses in details whose id does not appear in sales.
#       .isin() checks for each value of a column whether it appears in another column,
#       and ~ turns True into False and False into True.
print("sales without house details:", (~sales["house_id"].isin(details["id"])).sum())
print("houses without a sale:      ", (~details["id"].isin(sales["house_id"])).sum())

sales without house details: 0
houses without a sale:       0


Each row of the combined table should be one sale with its price, because the price is what you predict.

**Your answer:** Your counts show whether the two tables match completely. Does the type of merge change the result here? Which type would you choose if a house were missing from `details`, and why?

...

## Combine the Two Tables

Merge `details` onto `sales` with the type of merge you chose, using the key columns from question 1: `left_on` is the key in `sales`, `right_on` the key in `details`.

The `assert` at the end checks that the combined table has one row per sale, and stops with an error if it does not.

In [10]:
# TODO: merge details onto sales on the two key columns, with the type of merge you chose
df = sales.merge(
    details,
    left_on="house_id",
    right_on="id",
    how="left",
    suffixes=("", "_details"),
)

print(f"combined: {df.shape[0]} rows, {df.shape[1]} columns")
assert len(df) == len(sales), "The combined table should have exactly one row per sale"
df.head()

combined: 21597 rows, 23 columns


,date,price,house_id,id,id_details,bedrooms,bathrooms,sqft_living,sqft_lot,floors,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,1,7129300520,3.0,1.00,1180.0,5650.0,1.0,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,2,6414100192,3.0,2.25,2570.0,7242.0,2.0,...,7,2170.0,400.0,1951,1991.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,3,5631500400,2.0,1.00,770.0,10000.0,1.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4,2487200875,4.0,3.00,1960.0,5000.0,1.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,5,1954400510,3.0,2.00,1680.0,8080.0,1.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


## Save the Combined Data

The project notebook reads `data/eda.csv`, so write the combined table there once. `02_king_county_project.ipynb` then loads it directly instead of repeating the merge.

In [ ]:
# TODO: save the combined DataFrame for exploratory data analysis
df.to_csv("data/eda.csv", index=False)

print("Saved data/eda.csv")

21597 rows, 23 columns
Saved data/eda.csv


In [12]:
# Read the file back to check that all rows and columns were saved
check = pd.read_csv("data/eda.csv")
print(f"{check.shape[0]} rows, {check.shape[1]} columns")
check.head()

21597 rows, 23 columns


,date,price,house_id,id,id_details,bedrooms,bathrooms,sqft_living,sqft_lot,floors,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,1,7129300520,3.0,1.00,1180.0,5650.0,1.0,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,2,6414100192,3.0,2.25,2570.0,7242.0,2.0,...,7,2170.0,400.0,1951,1991.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,3,5631500400,2.0,1.00,770.0,10000.0,1.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4,2487200875,4.0,3.00,1960.0,5000.0,1.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,5,1954400510,3.0,2.00,1680.0,8080.0,1.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0
